In [5]:
import os
import sys
from os.path import join
import copy

# Add parent directory (arch/) to path so we can import sibling modules
# Handle both cases: notebook run from diagnostics/ or from arch/
current_dir = os.getcwd()
if 'diagnostics' in current_dir:
    arch_dir = os.path.dirname(current_dir)
else:
    arch_dir = current_dir

if arch_dir not in sys.path:
    sys.path.insert(0, arch_dir)
    
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import optim, nn
import torch.multiprocessing as mp
from torch.distributed import init_process_group, destroy_process_group
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from astropy.io import fits
import pyxis.torch as pxt
import normflows as nf

from networks import *
from train import *
import config
from model_registry import load_model_config

model_name = 'ViT-CNN-flow'
train_dir = '/ocean/projects/phy250048p/shared/datasets/valid_1m'
test_dir = '/ocean/projects/phy250048p/shared/datasets/valid_1m'
fig_dir = '/ocean/projects/phy250048p/shared/figures/'
model_dir = '/ocean/projects/phy250048p/shared/models/'

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [7]:
model_cfg = load_model_config(model_name, allow_fallback_current=True)
config.set_model_config(model_cfg)

KeyError: 'pretrain'

In [ ]:
model = load_model(
    train_config=config.train,
    Model=ForkCNN,
    path=join(model_dir, model_name, f'{model_name}199'),
    strict=True,
    assign=True,
    device=device,
    model_name=model_name,
    networks_root='/ocean/projects/phy250048p/shared/networks',
    use_compile=False,
)

RuntimeError: Error(s) in loading state_dict for ForkCNN:
	Missing key(s) in state_dict: "fully_connected_layer.0.weight", "fully_connected_layer.0.bias". 
	Unexpected key(s) in state_dict: "layer_norm.weight", "layer_norm.bias", "base._context_encoder.mlp.0.weight", "base._context_encoder.mlp.0.bias", "base._context_encoder.mlp.2.weight", "base._context_encoder.mlp.2.bias", "base._context_encoder.mlp.4.weight", "base._context_encoder.mlp.4.bias", "transform._transforms.0._permutation", "transform._transforms.1.autoregressive_net.initial_layer.weight", "transform._transforms.1.autoregressive_net.initial_layer.bias", "transform._transforms.1.autoregressive_net.initial_layer.mask", "transform._transforms.1.autoregressive_net.initial_layer.degrees", "transform._transforms.1.autoregressive_net.context_layer.weight", "transform._transforms.1.autoregressive_net.context_layer.bias", "transform._transforms.1.autoregressive_net.blocks.0.context_layer.weight", "transform._transforms.1.autoregres